# UC-Bench v0 — the complete workflow

**Purpose:** explain exactly what exists, what the agent sees, what it does, what stays private, how reward works, and what a legitimate end result looks like.

> Current state: Stages 1–6 pass. The Verifiers tool adapter and deterministic runtime smoke pass. No real model ranking exists yet because model endpoints and an isolated coding runtime are not configured. The report intentionally says **PRE-RESULT**.

## 1. The one-sentence product

UC-Bench tests whether a tool-using agent can make a defensible biomarker-development decision when biological evidence is small, shifted, contaminated, sufficient, recoverable, or irrecoverable.

The predictor is an artifact inside the task. The evaluated object is the **agent trajectory**: checks run, files produced, irreversible choices, calibration, containment, and next action.

## 2. System schematic

<svg width="100%" viewBox="0 0 1100 330" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="UC-Bench boundary schematic">
<defs><marker id="a" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto"><path d="M0,0 L8,4 L0,8 z" fill="#66758f"/></marker></defs>
<rect x="20" y="28" width="650" height="270" rx="18" fill="#eef5ff" stroke="#3874c8"/><text x="45" y="62" font-size="20" font-weight="700" fill="#173a6a">Agent-visible isolated coding workspace</text>
<rect x="50" y="88" width="170" height="72" rx="10" fill="white" stroke="#8ca8ce"/><text x="72" y="120" font-size="16">TASK.md + schemas</text><text x="72" y="143" font-size="13" fill="#59677d">ordinary files</text>
<rect x="255" y="88" width="180" height="72" rx="10" fill="white" stroke="#8ca8ce"/><text x="280" y="120" font-size="16">Development data</text><text x="280" y="143" font-size="13" fill="#59677d">full or withheld</text>
<rect x="470" y="88" width="165" height="72" rx="10" fill="white" stroke="#8ca8ce"/><text x="492" y="120" font-size="16">Agent artifacts</text><text x="492" y="143" font-size="13" fill="#59677d">JSON + .py evidence</text>
<rect x="155" y="202" width="400" height="62" rx="12" fill="#173a6a"/><text x="185" y="239" font-size="16" fill="white">ask analyst · commit · reveal · submit</text>
<rect x="745" y="28" width="335" height="270" rx="18" fill="#fff1f1" stroke="#bd3b4a"/><text x="770" y="62" font-size="20" font-weight="700" fill="#7e1e2a">Private grader boundary</text>
<rect x="780" y="90" width="265" height="58" rx="10" fill="white" stroke="#d58c95"/><text x="809" y="125" font-size="15">GSE92415 expression + labels</text>
<rect x="780" y="165" width="265" height="58" rx="10" fill="white" stroke="#d58c95"/><text x="822" y="200" font-size="15">variants + ground truth</text>
<rect x="780" y="240" width="265" height="38" rx="10" fill="#7e1e2a"/><text x="814" y="265" font-size="14" fill="white">aggregate result only</text>
<path d="M670 232 L745 232" stroke="#66758f" stroke-width="3" marker-end="url(#a)"/><path d="M745 260 L670 260" stroke="#66758f" stroke-width="3" marker-end="url(#a)"/>
</svg>

## 3. What exists now

| Layer | Concrete artifact | Status |
|---|---|---|
| Data | Three pinned GEO series matrices and three platform annotations | verified |
| Cohorts | 24 discovery, 23 untouched replication, 59 sealed transfer patients | verified |
| Start states | `full_data` and matched `data_withheld` packages | leakage-scanned |
| Predictor | Safe JSON logistic model over 1–50 within-sample gene ranks | executable |
| State | working → committed → revealed → submitted | tested |
| Private reveal | AUC, bootstrap interval, permutation p, n only | tested |
| Reward | Five evidence-backed components; no prose similarity | tested |
| Variants | Four symmetric states and five strength ladders | control-tested |
| Runtime | Verifiers 0.3.1 four-tool adapter | fixture smoke passed |
| Isolation | Docker, Prime, or Modal | **not configured** |
| Models | three models × three smoke attempts | **not run** |

In [1]:
import json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
status = json.loads((ROOT / 'audit/status.json').read_text())
[(stage['id'], stage['status'], stage['name']) for stage in status['stages']]

[(1, 'complete', 'Product and benchmark contract'),
 (2, 'complete', 'Data feasibility and provenance'),
 (3, 'complete', 'Expert reference episode'),
 (4, 'complete', 'Environment and tool contract'),
 (5, 'complete', 'Graders and graceful-failure reward'),
 (6, 'complete', 'Evidence states and difficulty ladders'),
 (7, 'in_progress', 'Agent integration and smoke pilot'),
 (8, 'not_started', 'Repeated evaluation and robustness'),
 (9, 'not_started', 'Leaderboard and failure atlas'),
 (10, 'not_started', 'Release audit and go/no-go')]

## 4. Biological substrate

| Role | Cohort | Baseline n | Drug | Endpoint | Platform |
|---|---|---:|---|---|---|
| Fit / optimistic sanity | GSE16879 | 24 | infliximab | complete mucosal healing, week 4–6 | GPL570 |
| Untouched development replication | GSE73661 | 23 | infliximab | endoscopic mucosal healing, week 4–6 | GPL6244 |
| Sealed transfer | GSE92415 | 59 | golimumab | clinical response, week 6 | GPL13158 |

This is intentionally a hard transfer: different drug, endpoint, study, and array. The v0 matrices are cohort-native normalized GEO series matrices mapped to 17,151 shared unambiguous genes. v0 does **not** pretend to re-normalize raw CEL files.

## 5. What the agent actually does

1. Audits files and reconstructs patient cohorts.
2. Checks baseline/timepoint/treatment/endpoint integrity.
3. Uses platform-robust features and preserves an untouched replication cohort.
4. Runs uncertainty and negative controls.
5. Writes an inspectable predictor, sample-role manifest, and structured analysis evidence.
6. Precommits expected AUC and interval; committed files are hashed.
7. Receives one aggregate private validation.
8. Diagnoses what failed or held, chooses `advance`, `stop`, or `insufficient_evidence`, and gives the smallest resolving next action.

The agent gets native coding tools inside an isolated workspace. Only private or irreversible transitions are custom tools: `ask_analyst`, `commit_analysis`, `reveal_validation`, and `submit`.

## 6. Irreversible episode

```text
WORKING
  │ commit_analysis(model + manifest + evidence + expectation)
  ▼
COMMITTED ── any file mutation ⇒ invalid episode
  │ reveal_validation() exactly once
  ▼
REVEALED ── only {AUC, bootstrap CI, permutation p, n}
  │ submit(final decision + codes + evidence + next action)
  ▼
SUBMITTED
```

The grader executes no agent pickle, joblib, or arbitrary model code. It applies the committed JSON coefficients privately.

## 7. Real reference episode

A deliberately small five-gene literature-fixed panel is represented by within-sample percentile ranks. It is not advertised as a clinical biomarker. It proves that the task is executable and establishes the honest decision regime.

In [2]:
reference = json.loads((ROOT / 'artifacts/reference/environment_episode.json').read_text())
{
    'phase': reference['phase'],
    'expected_auc': reference['commitment']['expected_auc'],
    'validation': reference['validation_result'],
    'decision': reference['submission']['decision'],
    'diagnostics': reference['submission']['diagnostic_codes'],
}

{'phase': 'submitted',
 'expected_auc': 0.6666666666666667,
 'validation': {'auc': 0.6666666666666667,
  'auc_interval': [0.5127314814814815, 0.8136574074074074],
  'evaluated_n': 59,
  'permutation_p_value': 0.014698530146985302},
 'decision': 'insufficient_evidence',
 'diagnostics': ['underpowered_validation',
  'endpoint_mismatch',
  'platform_shift',
  'drug_transfer']}

The authentic result is AUC 0.667, 95% bootstrap interval 0.513–0.814, permutation p=0.0147, n=59. It shows directional signal but misses the frozen AUC ≥0.70 advancement threshold and remains imprecise across endpoint/platform/drug transfer. The correct v0 decision is `insufficient_evidence`.

## 8. Reward without keyword hacking

| Component | Weight | Observable evidence |
|---|---:|---|
| Failure detection and diagnosis | 25% | supported diagnostic codes, not caveat prose |
| Evidence integrity and containment | 25% | event order, hashes, artifact shapes, recomputed evidence |
| Uncertainty and decision calibration | 20% | expected-vs-realized AUC, interval, correct decision/confidence |
| Recovery and next action | 15% | state-specific typed action |
| Reproducibility and contract | 15% | independently verified model/data audits and frozen settings |

Agent-authored `status: pass` fields are not trusted: cohort counts, sample roles, AUCs, intervals, permutation results, and gene coverage are recomputed from the committed model and workspace. A contradicted pass claim is capped at 40. Hard-invalid behavior scores zero. Raw AUC is never the headline model rank.

In [3]:
separation = json.loads((ROOT / 'artifacts/grading/reference_separation.json').read_text())
{
    'expert': separation['expert']['score'],
    'same-result mediocre': separation['mediocre']['score'],
    'keyword-only caution': separation['keyword_only_control']['score'],
    'structured false-pass claims': separation['structured_forgery_control']['score'],
    'sealed-outcome access': separation['hard_invalidation_control']['score'],
}

{'expert': 100.0,
 'same-result mediocre': 40.0,
 'keyword-only caution': 40.0,
 'structured false-pass claims': 40.0,
 'sealed-outcome access': 0.0}

## 9. Graceful failure needs symmetric states

| State | Correct behavior |
|---|---|
| Authentic weak transfer | preserve signal; `insufficient_evidence` |
| Controlled sufficient evidence | `advance`; abstention loses |
| Recoverable feature dropout | detect, repair, then decide |
| Irrecoverable sample identity conflict | contain and `stop` |

Five private ladders vary confounding, label noise, swaps, duplication, and feature dropout. Controlled states measure agent behavior only and are never reported as UC biology.

In [4]:
variants = json.loads((ROOT / 'artifacts/variants/scenario_controls.json').read_text())
variants['policy_control_scores']

{'always_advance': 35.0,
 'always_insufficient': 15.0,
 'always_stop': 15.0,
 'auc_chaser': 35.0,
 'state_aware_reference': 100.0}

## 10. What end results should look like

The finished public surface has: a repeated-model score with clustered uncertainty; contract-valid rate; decision confusion; calibration plot; full-data versus data-withheld retention; breaking curves; and short replayable examples of competent recovery, reckless success, and empty caution.

It does **not** show a model leaderboard until repeated real trajectories exist. Run `make report`, then open `reports/generated/index.html` for the current pre-result dashboard.

## 11. Where the engineer's predictor fits

A separately trained anti-TNF predictor can use the same `model.json` + `predictor_manifest.json` contract only if it is exactly exportable to the supported linear-rank form. A more complex model must run behind a separately isolated, immutable prediction adapter; the grader must never load its arbitrary pickle or code. It becomes an artifact the agent must diligence. The useful ablation is:

```text
agent alone        specialist predictor alone        agent + specialist predictor
      └──────────────────────┬──────────────────────────────┘
                    same sealed evaluator
```

This does not replace the environment. It tests whether the agent checks training provenance, endpoint fit, leakage, calibration, and uncertainty rather than blindly trusting a stronger model. The predictor must not have seen sealed GSE92415 outcomes.

## 12. Reproduce the current state

```bash
make status
make check
make reference
make environment
make grade-reference
make variants
make runtime-smoke
make report
make pre-release-audit
# or rebuild and verify the whole completed core:
make verify-v0
```

No JupyterLab service is required. This `.ipynb` opens in Codex, VS Code, or another notebook editor; reusable logic lives in tested `.py` files.

## 13. Honest completion boundary

The environment core, real-data reference, private boundary, grader controls, symmetric variants, repeated-evaluation statistics, and report shell exist. A real model ranking does not. The automated pre-release audit currently passes 7 of 11 checks and returns `no_go`. Stage 7 needs three configured model endpoints and a genuinely isolated coding runtime; Stage 8 then needs ten trajectories per model per headline condition. Until those gates pass and an independent reviewer signs off, a public release is a **no-go**, not a demo dressed up as a benchmark.